# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FatimaNdeem/Flyrank-ml-internship./blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!git clone "https://github.com/FatimaNdeem/Flyrank-ml-internship." /content/Flyrank-ml-internship.

fatal: destination path '/content/Flyrank-ml-internship.' already exists and is not an empty directory.


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'Flyrank-ml-internship.', 'sample_data']


In [ ]:
%cd /content/Flyrank-ml-internship.

/content/Flyrank-ml-internship.


In [ ]:
!ls -lh data/raw/

total 6.5M
-rw-r--r-- 1 root root 6.5M Aug 20 00:59 content_refresh_anonymized.csv


In [12]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Make sure we are inside the repository
%cd /content/Flyrank-ml-internship.

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("Number of columns:", len(df.columns))

/content/Flyrank-ml-internship.
Dataset shape: (30000, 44)
Number of columns: 44


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

I chose Random Forest because this is a classification problem where the goal is to identify content that is declining.

Random Forest can capture non-linear relationships between the available content, traffic, engagement, freshness, and search-demand features without requiring strong assumptions about their relationships.

It also provides feature importance measures that help interpret which observed signals the model relies on.

The model will be evaluated against the Week-4 baseline using the same grouped test set and the same evaluation metrics.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I use a grouped train/test split by client so that content from the same client does not appear in both the training and test sets.

This provides a stricter test of whether the model can generalize to unseen clients rather than learning client-specific patterns.

I use an 80/20 split with a fixed random state for reproducibility.

The target is defined as declining when impressions in the last 30 days are less than 80% of impressions in the previous 30 days.

In [13]:
# --------------------------------------------------
# Create target
# --------------------------------------------------

df["is_declining"] = (
    df["impressions_last_30d"]
    < 0.8 * df["impressions_prev_30d"]
).astype(int)

print("Dataset rows:", len(df))

print("\nTarget distribution:")
print(df["is_declining"].value_counts())


# --------------------------------------------------
# Grouped train/test split
# --------------------------------------------------

groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        df,
        df["is_declining"],
        groups=groups
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()


# --------------------------------------------------
# Check split
# --------------------------------------------------

print("\nTrain rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTrain clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

overlap = (
    set(train_df["client_id"])
    & set(test_df["client_id"])
)

print("Client overlap:", len(overlap))

Dataset rows: 30000

Target distribution:
is_declining
1    16262
0    13738
Name: count, dtype: int64

Train rows: 23837
Test rows: 6163

Train clients: 25
Test clients: 7
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

I train a Random Forest classifier using content, traffic, engagement, freshness, and search-demand features.

The two impression-window fields used directly to define the `is_declining` target are excluded from the model features. This reduces direct target leakage and provides a more conservative test of whether other observed signals can identify declining content.

The Random Forest is compared with the Week-4 baseline on the same grouped test rows and using the same evaluation metrics.

In [14]:
# --------------------------------------------------
# 1. Feature selection
# --------------------------------------------------

# Numeric features
#
# IMPORTANT:
# impressions_last_30d and impressions_prev_30d
# are excluded because they directly define the target.

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]


# Categorical features

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier"
]


# --------------------------------------------------
# 2. Build feature matrix
# --------------------------------------------------

X = df[
    numeric_features + categorical_features
].copy()


# Fill missing numeric values

for col in numeric_features:
    X[col] = X[col].fillna(
        X[col].median()
    )


# Fill missing categorical values

for col in categorical_features:
    X[col] = X[col].fillna("Unknown")


# Convert categorical variables to dummy variables

X = pd.get_dummies(
    X,
    columns=categorical_features,
    dtype=int
)

print("Feature matrix shape:", X.shape)


# --------------------------------------------------
# 3. Train/test feature split
# --------------------------------------------------

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = df["is_declining"].iloc[train_idx]
y_test = df["is_declining"].iloc[test_idx]


print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())


# --------------------------------------------------
# 4. Train Random Forest
# --------------------------------------------------

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(
    X_train,
    y_train
)


# --------------------------------------------------
# 5. Model predictions
# --------------------------------------------------

model_pred = model.predict(X_test)


# --------------------------------------------------
# 6. Random Forest metrics
# --------------------------------------------------

model_metrics = {
    "Accuracy": accuracy_score(
        y_test,
        model_pred
    ),

    "Precision": precision_score(
        y_test,
        model_pred,
        zero_division=0
    ),

    "Recall": recall_score(
        y_test,
        model_pred,
        zero_division=0
    ),

    "F1": f1_score(
        y_test,
        model_pred,
        zero_division=0
    )
}


print("\nRandom Forest metrics:")

for metric, value in model_metrics.items():
    print(
        f"{metric}: {value:.4f}"
    )

Feature matrix shape: (30000, 61)
X_train shape: (23837, 61)
X_test shape: (6163, 61)

Training target distribution:
is_declining
1    13113
0    10724
Name: count, dtype: int64

Testing target distribution:
is_declining
1    3149
0    3014
Name: count, dtype: int64

Random Forest metrics:
Accuracy: 0.5768
Precision: 0.5766
Recall: 0.6466
F1: 0.6096


In [ ]:
# Apply the Week-4 baseline logic to the same test rows

baseline_test = test_df.copy()

baseline_test["decline_score"] = (
    -baseline_test["trend_pct"]
).clip(lower=0)

baseline_test["freshness_score"] = (
    baseline_test["days_since_last_update"]
)

baseline_test["search_score"] = (
    baseline_test["search_volume"]
)

baseline_test["decline_score"] = baseline_test["decline_score"].fillna(0)
baseline_test["freshness_score"] = baseline_test["freshness_score"].fillna(0)
baseline_test["search_score"] = baseline_test["search_score"].fillna(0)

baseline_test["action_score"] = (
    0.5 * baseline_test["decline_score"]
    + 0.3 * baseline_test["freshness_score"]
    + 0.2 * baseline_test["search_score"]
)

# Same 75th-percentile rule used in ML-07,
# calculated from the training portion to avoid using test information.
threshold = (
    train_df["action_score"]
    if "action_score" in train_df.columns
    else None
)

if threshold is None:
    train_baseline = train_df.copy()

    train_baseline["decline_score"] = (
        -train_baseline["trend_pct"]
    ).clip(lower=0)

    train_baseline["freshness_score"] = (
        train_baseline["days_since_last_update"]
    ).fillna(0)

    train_baseline["search_score"] = (
        train_baseline["search_volume"]
    ).fillna(0)

    train_baseline["decline_score"] = (
        train_baseline["decline_score"].fillna(0)
    )

    train_baseline["action_score"] = (
        0.5 * train_baseline["decline_score"]
        + 0.3 * train_baseline["freshness_score"]
        + 0.2 * train_baseline["search_score"]
    )

    threshold = train_baseline["action_score"].quantile(0.75)
else:
    threshold = train_df["action_score"].quantile(0.75)

# Baseline prediction
baseline_pred = (
    baseline_test["action_score"] >= threshold
).astype(int)

baseline_f1 = f1_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_precision = precision_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_pred,
    zero_division=0
)

comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Random Forest"
    ],
    "Precision": [
        baseline_precision,
        model_metrics["Precision"]
    ],
    "Recall": [
        baseline_recall,
        model_metrics["Recall"]
    ],
    "F1": [
        baseline_f1,
        model_metrics["F1"]
    ]
})

comparison

,Method,Precision,Recall,F1
0,Week-4 Baseline,0.739373,0.336932,0.462914
1,Random Forest,0.576607,0.646554,0.609581


The Random Forest achieved a higher F1 and recall than the Week-4 baseline, while the baseline achieved higher precision and accuracy. This means the Random Forest identified more declining content, but also produced more false positives. The choice therefore depends on whether missing declining content or avoiding unnecessary refresh recommendations is more important

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The Random Forest was evaluated against the Week-4 baseline on the same grouped test set.

The two impression-window fields that directly define the target were excluded from the model features. This provides a more conservative test of whether other observed content, traffic, engagement, freshness, and search-demand signals can identify declining content.

The model should be treated as decision-support rather than as an automatic refresh decision.

Feature importance is reviewed to understand which observed signals the model relies on.

In [15]:
# --------------------------------------------------
# Feature importance
# --------------------------------------------------

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
})


feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
)


print("Top 10 features:")

feature_importance.head(10)

Top 10 features:


,feature,importance
5,impressions_90d,0.088246
18,avg_position,0.085944
13,days_with_impressions,0.075495
15,content_age_days,0.063604
4,char_count,0.047369
3,word_count,0.045862
17,ctr,0.043527
7,pageviews_90d,0.041548
20,scroll_rate,0.039869
8,sessions_90d,0.038219


In [16]:
# --------------------------------------------------
# Error analysis
# --------------------------------------------------

error_analysis = test_df[
    ["content_id", "is_declining"]
].copy()


# Add predictions

error_analysis["prediction"] = model_pred


# Default classification

error_analysis["error_type"] = "Correct"


# False negatives

error_analysis.loc[
    (error_analysis["is_declining"] == 1)
    & (error_analysis["prediction"] == 0),
    "error_type"
] = "False Negative"


# False positives

error_analysis.loc[
    (error_analysis["is_declining"] == 0)
    & (error_analysis["prediction"] == 1),
    "error_type"
] = "False Positive"


# Count errors

print("Error counts:")

print(
    error_analysis[
        "error_type"
    ].value_counts()
)


# Show false negatives

print("\nFalse negatives:")

print(
    error_analysis[
        error_analysis["error_type"]
        == "False Negative"
    ].head(5)
)


# Show false positives

print("\nFalse positives:")

print(
    error_analysis[
        error_analysis["error_type"]
        == "False Positive"
    ].head(5)
)

Error counts:
error_type
Correct           3555
False Positive    1495
False Negative    1113
Name: count, dtype: int64

False negatives:
              content_id  is_declining  prediction      error_type
1   content_a1fb4e703a9e             1           0  False Negative
23  content_2da6ae9d0882             1           0  False Negative
25  content_033ae3e7aecf             1           0  False Negative
39  content_4595e8704e07             1           0  False Negative
47  content_40cb4af260c0             1           0  False Negative

False positives:
              content_id  is_declining  prediction      error_type
13  content_a5a2fbc76336             0           1  False Positive
26  content_72c5c2d73e5a             0           1  False Positive
36  content_bce275871a25             0           1  False Positive
64  content_685de0e3b7cb             0           1  False Positive
78  content_dea0d86223f3             0           1  False Positive


### Error interpretation

The Random Forest made 1,495 false-positive predictions and 1,113 false-negative predictions on the grouped test set.

False positives are cases predicted as declining when the observed target was not declining. False negatives are declining cases that the model did not identify.

The model produced more false positives than false negatives, showing that it favors identifying more potentially declining content at the cost of additional incorrect alerts.

Compared with the Week-4 baseline, the Random Forest achieved higher recall and F1 but lower precision and accuracy. This means the model captured more of the observed declining cases, but its recommendations were less precise.

Feature importance showed that impressions_90d, avg_position, days_with_impressions, and content_age_days were among the strongest observed signals used by the model.

These results are directional and should be treated as decision-support rather than proof that a specific piece of content must be refreshed.

## Leakage check

The two fields used directly to construct `is_declining` — `impressions_last_30d` and `impressions_prev_30d` — were excluded from the Random Forest feature list.

The model features therefore do not directly contain the target definition.

The grouped train/test split also prevents the same client from appearing in both training and testing data.

No future outcome columns were intentionally included in the feature list.

In [17]:
# --------------------------------------------------
# Leakage check
# --------------------------------------------------

target_definition_features = [
    "impressions_last_30d",
    "impressions_prev_30d"
]


used_target_definition_features = [
    col
    for col in target_definition_features
    if col in numeric_features
]


future_like = [
    col
    for col in numeric_features + categorical_features
    if any(
        term in col.lower()
        for term in [
            "future",
            "next_30d",
            "next_60d",
            "next_90d"
        ]
    )
]


print(
    "Target-definition features used:",
    used_target_definition_features
)

print(
    "Future-like features used:",
    future_like
)

print(
    "Client overlap:",
    len(
        set(train_df["client_id"])
        & set(test_df["client_id"])
    )
)


leakage_free = (
    len(used_target_definition_features) == 0
    and len(future_like) == 0
    and len(
        set(train_df["client_id"])
        & set(test_df["client_id"])
    ) == 0
)


print(
    "\nLeakage check passed:",
    leakage_free
)

Target-definition features used: []
Future-like features used: []
Client overlap: 0

Leakage check passed: True


The model made 702 false-positive predictions and 462 false-negative predictions on the test set. False positives are cases predicted as declining when the observed target was not declining, while false negatives are declining cases that the model did not identify. The model therefore missed fewer positive cases than the number of false positives it produced. Feature importance showed that impressions_prev_30d and impressions_last_30d were the strongest observed signals, followed by impressions_90d and avg_position. These results are directional and should be treated as decision-support rather than proof that content must be refreshed.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.